# CNN toàn cục–cục bộ — nghiên cứu phân loại ống hút

Chỉ dataset hiện có, 5 lớp; không chưng cất. Mặc định chạy pilot Train/Validation; chưa sinh kết luận Test. Attach `straw_global_local_source.zip` và ảnh gốc. Bật GPU. Giữ cặp torch/torchvision tương thích của Kaggle. ResNet18 pretrained cần Internet hoặc file weight chính thức đã cache.

In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, json, shutil
archives = list(Path('/kaggle/input').rglob('straw_global_local_source.zip'))
assert len(archives) == 1, 'Attach đúng một source zip hoặc đặt SOURCE_ZIP thủ công.'
SOURCE_ZIP = archives[0]
PROJECT = Path('/kaggle/working/straw-global-local')
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    for name in archive.namelist():
        member = Path(name)
        assert not member.is_absolute() and '..' not in member.parts, 'Unsafe zip member'
    archive.extractall(PROJECT)
sys.path.insert(0, str(PROJECT))
print(PROJECT)

In [ ]:
import torch, torchvision
assert torch.cuda.is_available(), 'Chọn accelerator GPU trong Kaggle Settings.'
print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))
# Nếu môi trường thiếu dependency, chạy pip với requirements; không thay cặp torch/torchvision.
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT/'requirements/straw-thesis.txt')], check=True)
# Offline: đặt PRETRAINED_FILE tới resnet18-f37072fd.pth chính thức rồi copy vào torch hub cache.
PRETRAINED_FILE = None
if PRETRAINED_FILE:
    destination = Path(torch.hub.get_dir())/'checkpoints'/'resnet18-f37072fd.pth'
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PRETRAINED_FILE, destination)

In [ ]:
candidates = list(Path('/kaggle/input').rglob('Straw Defects in Juice Box'))
assert len(candidates) == 1, 'Đặt RAW_ROOT thủ công nếu có nhiều bản dataset.'
RAW_ROOT = candidates[0].parent
PROTOCOL = PROJECT/'artifacts/straw_global_local/data_v1'
CACHE = PROJECT/'artifacts/straw_global_local/cache_v1'
PHASE = 'development'  # Đổi thành confirmatory sau pilot và review thực tế.
CONFIG = PROJECT/('configs/straw_global_local_pilot.json' if PHASE == 'development' else 'configs/straw_global_local.json')
OUTPUT = PROJECT/f'artifacts/straw_global_local/{PHASE}_v1'
def run(action, *extra):
    command = [sys.executable, '-m', 'src.straw_global_local.runner', action,
               '--config', str(CONFIG), '--protocol', str(PROTOCOL), '--cache', str(CACHE), '--output', str(OUTPUT)]
    subprocess.run(command + list(extra), cwd=PROJECT, check=True)
if not (PROTOCOL/'protocol.json').exists():
    run('prepare', '--image-root', str(RAW_ROOT))
run('cache', '--cache-workers', '4')
run('plan')

## Pilot hoặc thực nghiệm xác nhận

Pilot mặc định B0/B5, một seed, ba epoch. Chưa đủ đánh giá hiệu quả. Thử B1 và ghi đầy đủ các lần tuning trước khi khóa recipe. Confirmatory mặc định B0–B6, ba seed, tối đa 40 epoch/patience10. Giữ cùng code/config/output khi resume; sửa recipe cần output mới. `--run-id B0_s42` có thể dùng để chia lượt chạy.

In [ ]:
run('train')
run('report')

## Rà dữ liệu và tài liệu trước khi truy cập Test

Không sửa các cờ bên dưới thành True khi chưa hoàn thành việc thực tế và có bằng chứng. Mô tả định nghĩa nhãn, rà các cặp/nhóm liên quan, nguồn/điều kiện sử dụng và related work. Notebook không tự xác nhận chúng. Giữ nguyên Train/Val/Test và không dùng Test để sửa mô hình.

In [ ]:
REVIEW_EVIDENCE = None  # Ví dụ: dict đã được người rà hoàn thành, có reviewer/notes dẫn artifact.
if REVIEW_EVIDENCE is not None:
    from src.straw_global_local.data import write_json
    write_json(PROTOCOL/'review.json', REVIEW_EVIDENCE)
print((PROTOCOL/'review.json').read_text())

## Khóa lựa chọn và đánh giá xác nhận

Chỉ chạy khi PHASE là confirmatory, tất cả run hoàn tất và review có bằng chứng. Benchmark dùng ảnh Validation gốc. Freeze chọn family bằng Validation trung bình seed. Test chỉ đánh giá cấu hình đã khóa. Kết quả âm hoặc CI chứa 0 phải được báo đúng.

In [ ]:
RUN_CONFIRMATORY_EVALUATION = False
if RUN_CONFIRMATORY_EVALUATION:
    assert PHASE == 'confirmatory'
    run('benchmark', '--repeats', '30')
    run('freeze')
    run('test')
    run('report', '--bootstrap', '1000')
else:
    print('Chưa mở Test. Pilot/development chỉ dùng Train và Validation.')

In [ ]:
REPORT = OUTPUT/'report'
if (REPORT/'readiness.json').exists():
    print((REPORT/'readiness.json').read_text())
print('Lưu checkpoint, config, prediction và report trước khi kết thúc Kaggle session:', OUTPUT)